# 16 — Lakebase Agent State & Persistent Memory
## Self-managed Lakebase implementation for the Investment AI POC

### Production recommendation

For new Databricks agents, the preferred architecture is:

```text
AgentKit
   ↓
Managed Agent Sessions + Agent Memory
   ↓
Databricks-managed Lakebase
```

Our Free Edition workspace cannot provision those managed stores because of the workspace-tier scale-to-zero restriction.

So for this POC we will demonstrate the **self-managed Lakebase pattern** instead:

```text
Investment AI Agent
        ↓
Session + Memory logic
        ↓
Lakebase Postgres
        ↓
agent_sessions + agent_memory
```

This still uses real Lakebase. The difference is that **we own the schema and persistence logic** instead of AgentKit managing it for us.

The Lakebase project was created manually in the UI:

- Project: `investment-ai-agent-state`
- Branch: `production`
- Compute: `primary`
- Database: `databricks_postgres`


## 1. Install the Postgres client

We use the standard `psycopg` Postgres driver plus the Databricks SDK.

The OAuth database token is generated programmatically, so no password or one-hour token is hardcoded in this notebook.


In [0]:
# NOTE: install pure-Python psycopg, NOT psycopg[binary].
# The binary wheel bundles its own OpenSSL, which fails the FIPS self-test
# on this compute (SIGABRT at import). Pure psycopg loads the FIPS-compliant
# system libpq via ctypes instead.
%pip uninstall -q -y psycopg-binary
%pip install -q --upgrade "psycopg>=3.2" "databricks-sdk>=0.118.0" anthropic
dbutils.library.restartPython()


## 2. Discover Lakebase and connect with OAuth

Instead of copying the temporary OAuth token from the Lakebase UI, we use the Databricks SDK to:

1. locate the `primary` Lakebase compute,
2. discover its host,
3. determine the current Databricks identity,
4. generate a fresh database credential,
5. connect securely with SSL.

The OAuth role for the project owner is created automatically when the Lakebase project is created.


In [0]:
import psycopg
from databricks.sdk import WorkspaceClient

PROJECT_ID = "investment-ai-agent-state"
BRANCH_ID = "production"
ENDPOINT_ID = "primary"
DATABASE = "databricks_postgres"
PORT = 5432

PROJECT = f"projects/{PROJECT_ID}"
BRANCH = f"{PROJECT}/branches/{BRANCH_ID}"
ENDPOINT = f"{BRANCH}/endpoints/{ENDPOINT_ID}"

w = WorkspaceClient()

endpoint = w.postgres.get_endpoint(name=ENDPOINT)
PGHOST = endpoint.status.hosts.host

current_user = w.current_user.me()
PGUSER = current_user.user_name

print("Project :", PROJECT_ID)
print("Branch  :", BRANCH_ID)
print("Compute :", ENDPOINT_ID)
print("Database:", DATABASE)
print("Host    :", PGHOST)
print("OAuth role resolved from current Databricks identity.")


## 3. Create a reusable Lakebase connection helper

Lakebase OAuth database credentials expire after one hour.

For this notebook, the safest pattern is to mint a fresh credential whenever we open a new database connection.


In [0]:
def get_lakebase_connection():
    credential = w.postgres.generate_database_credential(
        endpoint=ENDPOINT
    )

    return psycopg.connect(
        host=PGHOST,
        port=PORT,
        dbname=DATABASE,
        user=PGUSER,
        password=credential.token,
        sslmode="require",
    )


with get_lakebase_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user, version();")
        database, db_user, version = cur.fetchone()

print("✅ Connected to Lakebase")
print("Database:", database)
print("Postgres user:", db_user)
print("Version:", version.split(",")[0])


## 4. Create the agent-state schema

Because this is the self-managed approach, we explicitly define the database schema.

We need two tables:

```text
agent_sessions
= ordered conversation messages inside one session

agent_memory
= durable facts/preferences that can survive across sessions
```

This is the main responsibility that AgentKit would normally manage for us.


In [0]:
CREATE_SCHEMA_SQL = '''
CREATE SCHEMA IF NOT EXISTS agent_state;

CREATE TABLE IF NOT EXISTS agent_state.agent_sessions (
    message_id      BIGSERIAL PRIMARY KEY,
    actor_id        TEXT NOT NULL,
    session_id      TEXT NOT NULL,
    role            TEXT NOT NULL CHECK (role IN ('user', 'assistant')),
    content         TEXT NOT NULL,
    created_at      TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_agent_sessions_actor_session
ON agent_state.agent_sessions(actor_id, session_id, message_id);

CREATE TABLE IF NOT EXISTS agent_state.agent_memory (
    memory_id          TEXT PRIMARY KEY,
    actor_id           TEXT NOT NULL,
    memory_type        TEXT NOT NULL,
    content            TEXT NOT NULL,
    source_session_id  TEXT,
    created_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    updated_at         TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_agent_memory_actor_type
ON agent_state.agent_memory(actor_id, memory_type);
'''

with get_lakebase_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(CREATE_SCHEMA_SQL)
    conn.commit()

print("✅ agent_state schema created")
print("✅ agent_sessions table ready")
print("✅ agent_memory table ready")


## 5. Verify the tables in Lakebase

After this cell runs, you can open:

```text
Lakebase Postgres
→ investment-ai-agent-state
→ production
→ Tables
→ agent_state
```

and see both tables directly in the Lakebase UI.


In [0]:
with get_lakebase_connection() as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT table_schema, table_name
            FROM information_schema.tables
            WHERE table_schema = 'agent_state'
            ORDER BY table_name
        ''')
        tables = cur.fetchall()

for schema_name, table_name in tables:
    print(f"{schema_name}.{table_name}")


## 6. Session persistence helpers

A session represents one conversation.

The important identifiers are:

```text
actor_id
= who owns the state

session_id
= one specific conversation
```

In production, `actor_id` should come from trusted authenticated application context.


In [0]:
def save_session_message(actor_id, session_id, role, content):
    with get_lakebase_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                '''
                INSERT INTO agent_state.agent_sessions
                    (actor_id, session_id, role, content)
                VALUES (%s, %s, %s, %s)
                ''',
                (actor_id, session_id, role, content),
            )
        conn.commit()


def load_session_history(actor_id, session_id):
    with get_lakebase_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                '''
                SELECT role, content
                FROM agent_state.agent_sessions
                WHERE actor_id = %s
                  AND session_id = %s
                ORDER BY message_id
                ''',
                (actor_id, session_id),
            )
            rows = cur.fetchall()

    return [
        {"role": role, "content": content}
        for role, content in rows
    ]


## 7. Long-term memory helpers

For this POC, we keep memory deliberately simple and governed.

A durable memory has:

- an owner (`actor_id`),
- a type,
- the remembered content,
- optionally the session where it came from.

We upsert by actor + memory type so a newer preference can replace an older one.


In [0]:
from uuid import uuid4

def upsert_memory(
    actor_id,
    memory_type,
    content,
    source_session_id=None,
):
    with get_lakebase_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                '''
                SELECT memory_id
                FROM agent_state.agent_memory
                WHERE actor_id = %s
                  AND memory_type = %s
                LIMIT 1
                ''',
                (actor_id, memory_type),
            )
            existing = cur.fetchone()

            if existing:
                cur.execute(
                    '''
                    UPDATE agent_state.agent_memory
                    SET content = %s,
                        source_session_id = %s,
                        updated_at = NOW()
                    WHERE memory_id = %s
                    ''',
                    (content, source_session_id, existing[0]),
                )
            else:
                cur.execute(
                    '''
                    INSERT INTO agent_state.agent_memory
                        (memory_id, actor_id, memory_type, content, source_session_id)
                    VALUES (%s, %s, %s, %s, %s)
                    ''',
                    (
                        str(uuid4()),
                        actor_id,
                        memory_type,
                        content,
                        source_session_id,
                    ),
                )
        conn.commit()


def load_memories(actor_id):
    with get_lakebase_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                '''
                SELECT memory_type, content
                FROM agent_state.agent_memory
                WHERE actor_id = %s
                ORDER BY updated_at DESC
                ''',
                (actor_id,),
            )
            rows = cur.fetchall()

    return [
        {"memory_type": memory_type, "content": content}
        for memory_type, content in rows
    ]


## 8. Configure Claude through Unity Gateway

The memory layer should not replace the model architecture we already built.

Claude still runs through our governed Unity Gateway Model Provider Service.

Before each turn, the application loads:

```text
current session history
        +
durable user memory
        ↓
Claude
```

Then the new messages are written back to Lakebase.


In [0]:
from anthropic import Anthropic

MODEL_PROVIDER_SERVICE = "investment_ai.gateway.anthropic_claude"
MODEL = "claude-sonnet-5-5"

workspace_auth = w.config.authenticate()["Authorization"]
databricks_token = workspace_auth.replace("Bearer ", "", 1)

anthropic_client = Anthropic(
    auth_token=databricks_token,
    base_url=f"{w.config.host.rstrip('/')}/ai-gateway/anthropic",
    default_headers={
        "Databricks-Model-Provider-Service": MODEL_PROVIDER_SERVICE,
    },
)

BASE_SYSTEM_PROMPT = '''
You are an Investment AI assistant for portfolio managers.

Use the current conversation history to maintain continuity.
Use durable memories only when they are relevant to the current request.
Do not invent memories that are not supplied.

Keep responses concise and professional.
'''.strip()

print("✅ Claude configured through Unity Gateway")


## 9. Build the state-aware message handler

This is the core application pattern.

For every message:

```text
Load session history from Lakebase
        +
Load durable memory from Lakebase
        ↓
Call Claude
        ↓
Store user + assistant messages back in Lakebase
```

The application itself remains stateless; Lakebase holds the durable state.


In [0]:
def handle_message(actor_id, session_id, user_message):
    history = load_session_history(actor_id, session_id)
    memories = load_memories(actor_id)

    system_prompt = BASE_SYSTEM_PROMPT

    if memories:
        memory_text = "\n".join(
            f"- {m['memory_type']}: {m['content']}"
            for m in memories
        )
        system_prompt += (
            "\n\nRelevant durable memory from Lakebase:\n"
            + memory_text
        )

    response = anthropic_client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=system_prompt,
        messages=[
            *history,
            {"role": "user", "content": user_message},
        ],
    )

    reply = "".join(
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text"
    )

    save_session_message(
        actor_id, session_id, "user", user_message
    )
    save_session_message(
        actor_id, session_id, "assistant", reply
    )

    return reply


# Part A — Short-term state inside one conversation

First we demonstrate session state.

The first turn establishes context:

```text
Fund = Synthetic Growth Fund 01
Concentration threshold = 10%
```

Then we ask a follow-up without repeating either value.


In [0]:
ACTOR_ID = "portfolio-manager-demo"
SESSION_1 = f"portfolio-review-{uuid4().hex[:8]}"

message_1 = (
    "For this investment review, focus on Synthetic Growth Fund 01 "
    "and use a 10% concentration threshold."
)

reply_1 = handle_message(
    ACTOR_ID,
    SESSION_1,
    message_1,
)

print("SESSION:", SESSION_1)
print("USER:", message_1)
print("ASSISTANT:", reply_1)


## 10. Follow up using only the persisted session

This question does not contain the fund name or threshold.

Claude can answer because the earlier conversation is reloaded from Lakebase.


In [0]:
message_2 = (
    "Before we continue, remind me which fund and concentration "
    "threshold I asked you to use."
)

reply_2 = handle_message(
    ACTOR_ID,
    SESSION_1,
    message_2,
)

print("USER:", message_2)
print("ASSISTANT:", reply_2)


## 11. Inspect the session rows

This is the key proof that our state is actually persisted in Lakebase rather than only living in a Python variable.


In [0]:
with get_lakebase_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT
                message_id,
                session_id,
                role,
                content,
                created_at
            FROM agent_state.agent_sessions
            WHERE actor_id = %s
              AND session_id = %s
            ORDER BY message_id
            ''',
            (ACTOR_ID, SESSION_1),
        )
        rows = cur.fetchall()

for message_id, session_id, role, content, created_at in rows:
    print(
        f"message_id : {message_id}\n"
        f"session_id : {session_id}\n"
        f"role       : {role}\n"
        f"content    : {content}\n"
        f"created_at : {created_at}\n"
        f"{'-' * 60}"
    )


# Part B — Long-term memory across conversations

Session history should not be reused forever.

Some information is durable enough to survive beyond a single conversation.

For this POC, we store a portfolio-manager preference:

```text
Prefers concise, risk-first portfolio summaries.
Default concentration threshold = 10%.
```

This belongs in long-term memory rather than ordinary session history.


In [0]:
preference = (
    "The portfolio manager prefers concise, risk-first portfolio summaries. "
    "Use a 10% concentration threshold by default unless explicitly overridden."
)

upsert_memory(
    actor_id=ACTOR_ID,
    memory_type="portfolio_review_preference",
    content=preference,
    source_session_id=SESSION_1,
)

print("✅ Durable memory written to Lakebase")


## 12. Inspect the memory row in Lakebase

This memory is stored independently from the conversation transcript.

That separation is important:

```text
Session table
= what was said in one conversation

Memory table
= what should still matter later
```


In [0]:
with get_lakebase_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT
                memory_type,
                content,
                source_session_id,
                created_at,
                updated_at
            FROM agent_state.agent_memory
            WHERE actor_id = %s
            ORDER BY updated_at DESC
            ''',
            (ACTOR_ID,),
        )
        memories = cur.fetchall()

for memory_type, content, source_session_id, created_at, updated_at in memories:
    print(
        f"memory_type       : {memory_type}\n"
        f"content           : {content}\n"
        f"source_session_id : {source_session_id}\n"
        f"created_at        : {created_at}\n"
        f"updated_at        : {updated_at}\n"
        f"{'-' * 60}"
    )


## 13. Start a completely new session

Now we create a brand-new `session_id`.

There is no conversation history from the first session.

If Claude still knows the review style and default threshold, that information came from `agent_memory`, not from the previous conversation transcript.


In [0]:
SESSION_2 = f"portfolio-review-{uuid4().hex[:8]}"

print("Old session:", SESSION_1)
print("New session:", SESSION_2)
print("New session history:", load_session_history(ACTOR_ID, SESSION_2))

message_3 = (
    "I'm starting a new portfolio review. "
    "How should you structure the review for me, and what concentration "
    "threshold should you use by default?"
)

reply_3 = handle_message(
    ACTOR_ID,
    SESSION_2,
    message_3,
)

print("\nUSER:", message_3)
print("ASSISTANT:", reply_3)


## 14. Final Lakebase verification

At this point Lakebase should contain:

```text
agent_state.agent_sessions
├── Conversation 1 messages
└── Conversation 2 messages

agent_state.agent_memory
└── Durable portfolio-review preference
```

You can verify the same records directly from the Lakebase **Tables** or **SQL Editor** UI.


In [0]:
with get_lakebase_connection() as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT
                session_id,
                COUNT(*) AS message_count
            FROM agent_state.agent_sessions
            WHERE actor_id = %s
            GROUP BY session_id
            ORDER BY session_id
        ''', (ACTOR_ID,))
        session_counts = cur.fetchall()

        cur.execute('''
            SELECT COUNT(*)
            FROM agent_state.agent_memory
            WHERE actor_id = %s
        ''', (ACTOR_ID,))
        memory_count = cur.fetchone()[0]

print("Persisted sessions:")
for session_id, count in session_counts:
    print(f"  {session_id}: {count} messages")

print("Durable memories:", memory_count)


# ✅ Lakebase State & Memory complete

We have now demonstrated real persistent agent state in Lakebase.

### Short-term state

```text
Conversation 1
   ↓
agent_sessions
   ↓
Follow-up turn
   ↓
Earlier context restored
```

### Long-term memory

```text
Durable preference
   ↓
agent_memory
   ↓
Completely new conversation
   ↓
Preference recalled
```

### Important production distinction

For new Databricks agents, the preferred implementation is still:

```text
Managed Agent Sessions + Agent Memory
        ↓
Databricks-managed Lakebase
```

Our Free Edition workspace could not provision those managed stores, so this notebook uses the documented **self-managed Lakebase** pattern.

The behavior is similar, but in this version **we manage the Postgres schema, persistence, and retrieval logic ourselves**.

### Next

**Production Agent Engineering**

We will bring together prompt/version management, retries, timeouts, fallbacks, model controls, and cost-aware execution.
